# Project 11 — Local Website FAQ Bot
## Ingest One Website and Answer Questions Over It

**What you'll learn:**
- Crawl/scrape a website into documents
- Chunk HTML content intelligently
- Build a local RAG pipeline for website Q&A
- Handle noisy web content (nav, footers, boilerplate)

**Stack:** Ollama · LangChain · ChromaDB · BeautifulSoup · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb beautifulsoup4

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
print("Models ready!")

## Step 2 — Create Sample Website Content

In [ ]:
from pathlib import Path
import json

data_dir = Path("sample_data"); data_dir.mkdir(exist_ok=True)

# Simulated website pages (in practice, use WebBaseLoader or a crawler)
pages = [
    {"url": "/pricing", "title": "Pricing", "content": """
Our Pricing Plans:
- Starter: $9/mo — 1 user, 5GB storage, email support
- Professional: $29/mo — 5 users, 50GB storage, priority support, API access
- Enterprise: $99/mo — Unlimited users, 500GB, dedicated support, SSO, audit logs
All plans include a 14-day free trial. Annual billing saves 20%.
"""},
    {"url": "/features", "title": "Features", "content": """
Key Features:
- Real-time Collaboration: Edit documents simultaneously with your team
- AI-Powered Search: Find any document in seconds with semantic search
- Version History: Track all changes with unlimited version history
- Integrations: Connect with Slack, Jira, GitHub, and 50+ other tools
- Security: SOC 2 Type II certified, end-to-end encryption at rest and in transit
- API Access: RESTful API with comprehensive documentation
"""},
    {"url": "/faq", "title": "FAQ", "content": """
Frequently Asked Questions:
Q: Can I cancel anytime?
A: Yes, you can cancel your subscription at any time. No cancellation fees.

Q: Do you offer refunds?
A: We offer a full refund within the first 30 days of any paid plan.

Q: Is my data secure?
A: Yes. We use AES-256 encryption and are SOC 2 Type II certified.

Q: Can I export my data?
A: Yes, you can export all your data in standard formats (CSV, JSON, PDF).

Q: Do you support single sign-on (SSO)?
A: SSO is available on the Enterprise plan.
"""},
    {"url": "/about", "title": "About Us", "content": """
About TechDocs Inc.
Founded in 2020, TechDocs helps teams organize and search their documentation
efficiently. Our platform serves over 10,000 companies worldwide, from startups
to Fortune 500 enterprises. Headquartered in San Francisco with remote teams
across 15 countries.
"""},
]

(data_dir / "website_pages.json").write_text(json.dumps(pages, indent=2))
print(f"Created {len(pages)} simulated website pages")

## Step 3 — Load and Chunk Website Content

In [ ]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

documents = []
for page in pages:
    doc = Document(
        page_content=page["content"].strip(),
        metadata={"url": page["url"], "title": page["title"], "source": "website"}
    )
    documents.append(doc)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(documents)

print(f"Pages: {len(documents)} → Chunks: {len(chunks)}")
for c in chunks:
    print(f"  [{c.metadata['title']}] {len(c.page_content)} chars")

## Step 4 — Build Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings,
    persist_directory="sample_data/website_chroma",
    collection_name="website_faq",
)
print(f"Vector store created with {len(chunks)} chunks")

# Test retrieval
results = vectorstore.similarity_search("How much does the enterprise plan cost?", k=2)
for r in results:
    print(f"  [{r.metadata['title']}] {r.page_content[:80]}...")

## Step 5 — Build FAQ Bot Chain

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

faq_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful FAQ bot for TechDocs Inc. Answer the customer's
question using ONLY the website content provided. If the answer isn't in the
context, say "I don't have that information on our website. Please contact
support@techdocs.com for help."

Website Content:
{context}

Customer Question: {question}

Answer (be specific, cite page sections when possible):"""
)

faq_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": faq_prompt},
)
print("FAQ bot ready!")

## Step 6 — Test the FAQ Bot

In [ ]:
questions = [
    "How much does the Professional plan cost?",
    "Do you offer SSO?",
    "Can I get a refund if I cancel after 2 weeks?",
    "How many companies use TechDocs?",
    "What integrations do you support?",
    "Do you have a mobile app?",  # Not in the content
]

for q in questions:
    print(f"\nQ: {q}")
    result = faq_chain.invoke({"query": q})
    print(f"A: {result['result']}")
    sources = [s.metadata['title'] for s in result['source_documents']]
    print(f"Sources: {sources}")

## What You Learned
- **Website content ingestion** with metadata
- **Domain-specific RAG** for customer-facing FAQ bots
- **Graceful fallback** when info is not in the knowledge base